In [1]:
import fitz  # PyMuPDF
import io
from PIL import Image
import os
import re
from openpyxl import Workbook

def extract_images_and_text_from_pdf(pdf_path, output_folder):
    if not os.path.isfile(pdf_path):
        print(f"File not found: {pdf_path}")
        return

    pdf_document = fitz.open(pdf_path)
    os.makedirs(output_folder, exist_ok=True)

    date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')
    current_date_folder = "unknown_date"
    previous_date_folder = "unknown_date"
    previous_date_y = 0
    text_by_date = {}

    for page_num in range(len(pdf_document)):
        page = pdf_document[page_num]
        image_list = page.get_images(full=True)
        blocks = page.get_text("dict")["blocks"]

        page_dates = date_pattern.findall(page.get_text())
        if page_dates:
            current_date_folder = page_dates[0]
            previous_date_folder = current_date_folder
            previous_date_y = 0  # 페이지가 바뀌면 이전 날짜의 y좌표 초기화

        if current_date_folder not in text_by_date:
            text_by_date[current_date_folder] = ""

        # 텍스트 블록에서 날짜를 추출하고 y좌표를 기록
        date_positions = []
        for block in blocks:
            # 텍스트 추출
            if "lines" in block:
                block_text = ""
                for line in block["lines"]:
                    for span in line["spans"]:
                        block_text += span["text"]
                dates = date_pattern.findall(block_text)
                if dates:
                    y_position = block["bbox"][1]
                    date_positions.append((dates[-1], y_position))
                    previous_date_folder = current_date_folder
                    previous_date_y = y_position
                    current_date_folder = dates[-1]
                    if current_date_folder not in text_by_date:
                        text_by_date[current_date_folder] = ""

        # 디버깅을 위해 텍스트 블록과 날짜의 y좌표 출력
        print(f"Page {page_num + 1} text blocks with dates and their y-coordinates:")
        for date, y in date_positions:
            print(f"Date: {date}, y-coordinate: {y}")

        # 이미지 처리
        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = pdf_document.extract_image(xref)
            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes))

            # 이미지의 y좌표 가져오기
            img_bbox = None
            for block in blocks:
                if block.get("image") == xref:
                    img_bbox = block["bbox"]
                    break

            if img_bbox:
                image_y = img_bbox[1]

                # 디버깅을 위해 이미지의 y좌표 출력
                print(f"Page {page_num + 1}, Image {img_index + 1}, y-coordinate: {image_y}")

                # 이미지가 속할 날짜 폴더 결정
                assigned_date = previous_date_folder
                for date, y_position in date_positions:
                    if image_y > y_position:
                        assigned_date = date
                        break
            else:
                assigned_date = current_date_folder

            # 이미지 저장 폴더 생성 (이미 모든 이미지를 같은 폴더에 저장)
            page_output_folder = output_folder
            os.makedirs(page_output_folder, exist_ok=True)
        
            # 파일 이름 및 경로 지정
            image_filename = f"img_{page_num + 1}_{img_index + 1}.jpg"
            image_path = os.path.join(page_output_folder, image_filename)
        
            # 이미지 저장
            image.save(image_path)
            print(f"Saved image: {image_path}")

# Define the paths
pdf_path = "/Users/sinequanon/Documents/PerfectS/data/pdf_data/TEST_05.pdf"
output_folder = "/Users/sinequanon/Documents/PerfectS/data/images"

# Call the function to extract images and text
extract_images_and_text_from_pdf(pdf_path, output_folder)

Page 1 text blocks with dates and their y-coordinates:
Date: 2018-08-01, y-coordinate: 172.40345764160156
Date: 2023-03-13, y-coordinate: 307.1134033203125
Page 2 text blocks with dates and their y-coordinates:
Page 3 text blocks with dates and their y-coordinates:
Page 4 text blocks with dates and their y-coordinates:
Saved image: /Users/sinequanon/Documents/PerfectS/data/images/img_4_1.jpg
Saved image: /Users/sinequanon/Documents/PerfectS/data/images/img_4_2.jpg
Saved image: /Users/sinequanon/Documents/PerfectS/data/images/img_4_3.jpg
Saved image: /Users/sinequanon/Documents/PerfectS/data/images/img_4_4.jpg
Page 5 text blocks with dates and their y-coordinates:
Date: 2023-03-13, y-coordinate: 677.1234130859375
Saved image: /Users/sinequanon/Documents/PerfectS/data/images/img_5_1.jpg
Saved image: /Users/sinequanon/Documents/PerfectS/data/images/img_5_2.jpg
Saved image: /Users/sinequanon/Documents/PerfectS/data/images/img_5_3.jpg
Saved image: /Users/sinequanon/Documents/PerfectS/data/i

In [4]:
import os
import pandas as pd
from pptx import Presentation
from pptx.util import Inches, Cm, Pt
from main import *

#슬라이드에서 특정 도형rhk 택스트를 선택한다
def select_shape_by_text(slide, text):
    for shape in slide.shapes:
        if shape.has_text_frame and text in shape.text:
            return shape
    return None

#여러개의 텍스트 일치 비교
def select_shapes_by_texts(slide, texts):
    shapes_with_texts = []
    for shape in slide.shapes:
        if shape.has_text_frame:
            for text in texts:
                if text in shape.text:
                    shapes_with_texts.append(shape)
                    break  # 한 텍스트가 일치하면 다른 텍스트는 확인할 필요 없음
    return shapes_with_texts


def insert_data(slide, animal_data):
    signalment_shape = select_shape_by_text(slide, "Signalment_data")
    if signalment_shape:
        signalment_shape.text_frame.clear()
        signalment_text = (
            f"이름: {animal_data['name']}\n"
            f"품종: {animal_data['breed']}\n"
            f"성별: {animal_data['sex']}\n"
            f"종류: {animal_data['species']}\n"
            f"나이: {animal_data['birth']}\n"
            f"현재 무게: {animal_data['weight']}\n"
            f"보호자: {animal_data['protector']}"
        )
        signalment_shape.text_frame.text = signalment_text
def insert_soap(slide, soap_data):
    # Subjective 데이터 삽입
    subjective_shape = select_shape_by_text(slide, 'Subjective text')
    if subjective_shape:
        
        subjective_shape.text_frame.clear()
        #Subjective text에 추출된 데이터를 넣는다
        for paragraph in soap_data.get('subjective', []):
            p = subjective_shape.text_frame.add_paragraph()
            p.text = paragraph
       
    else:
        print("Subjective shape not found")

    # Objective 데이터 삽입
    objective_shape = select_shape_by_text(slide, 'Objective text')
    if objective_shape:
      #Objective text에 추출된 데이터를 넣는다
        objective_shape.text_frame.clear()
        for paragraph in soap_data.get('objective', []):
            p = subjective_shape.text_frame.add_paragraph()
            p.text = paragraph
    else:
        print("Objective shape not found")

    # Assessment 데이터 삽입
    assessment_shape = select_shape_by_text(slide, 'Assessment text')
    if assessment_shape:
        #Assessment text에 추출된 데이터를 넣는다
        assessment_shape.text_frame.clear()
        for paragraph in soap_data.get('assessment', []):
            p = assessment_shape.text_frame.add_paragraph()
            p.text = paragraph
    
    else:
        print("Assessment shape not found")

    # Plan 데이터 삽입
    plan_shape = select_shape_by_text(slide, 'Plan text')
    if plan_shape:
    #Plan text에 추출된 데이터를 넣는다
        plan_shape.text_frame.clear()
        for paragraph in soap_data.get('plan', []):
            p = plan_shape.text_frame.add_paragraph()
            p.text = paragraph

    else:
        print("Plan shape not found")
#특정 슬라이드 복사
def copy_slide(prs, slide_index):
    #prs - 프레젠테이션 , 
    #복사할 슬라이드와 레이아웃을 가져온다
    slide = prs.slides[slide_index]
    slide_layout = slide.slide_layout
    #새 슬라이드를 만든다
    new_slide = prs.slides.add_slide(slide_layout)

    for shape in slide.shapes:
        if not shape.has_text_frame:
            if shape.shape_type == 13: #도형이 이미지인지 확인
                #이미지 도형인 경우 이미지 파일로 저장한다
                image_path = '/Users/sinequanon/Documents/PerfectS/data/images/temp.png'
                with open(image_path, 'wb') as f:
                    f.write(shape.image.blob)
                    #텍스프 프레임이 있을 경우 동일한 위치와 크기로 새 슬라이드에 추가
                new_slide.shapes.add_picture(image_path, shape.left, shape.top, shape.width, shape.height)
            else:
                #이미지가 아닌 도형 복사
                new_shape = new_slide.shapes.add_shape(shape.auto_shape_type, shape.left, shape.top, shape.width, shape.height)
        else:
            #텍스트 프레임이 있는 도형 복사
            new_shape = new_slide.shapes.add_textbox(shape.left, shape.top, shape.width, shape.height)
            for paragraph in shape.text_frame.paragraphs:
                new_paragraph = new_shape.text_frame.add_paragraph()
                new_paragraph.text = paragraph.text

    return new_slide# 새로운 슬라이드 반환

#vital check 표 넣기
def add_vital_check_table_to_slide(prs, headers, data_list):
    if data_list is None or (isinstance(data_list, pd.DataFrame) and data_list.empty) or (isinstance(data_list, list) and not data_list):
        data_list = [{}]

    slide_layout = prs.slide_layouts[5]  # 빈 슬라이드 레이아웃 추가
    slide = prs.slides.add_slide(slide_layout)
    title = slide.shapes.title
    title.text = "Vital Check"

    rows = len(data_list) + 1  # 헤더를 포함한 행 수
    cols = len(headers)   # 헤더의 수
    #테이블 추가
    table = slide.shapes.add_table(rows, cols, Inches(0.5), Inches(1.5), Inches(9), Inches(5.0)).table
    #헤더 채우기
    for col_idx, header in enumerate(headers):
        table.cell(0, col_idx).text = header

    #데이터 채우기
    for row_idx, data in enumerate(data_list):
        for col_idx, header in enumerate(headers):
            table.cell(row_idx + 1, col_idx).text = str(data[header]) if isinstance(data, dict) else str(data)

#lab Result 채우기
def add_lab_table_to_slide(prs, headers, lab_data_list):
    if lab_data_list is None or (isinstance(lab_data_list, pd.DataFrame) and lab_data_list.empty) or (isinstance(lab_data_list, list) and not lab_data_list):
        lab_data_list = [{}]
    
    for lab_data in lab_data_list:
        if isinstance(lab_data, dict) and isinstance(lab_data.get('table'), pd.DataFrame):
            slide_layout = prs.slide_layouts[5]  #빈 슬라이드에 레이아웃 추가
            slide = prs.slides.add_slide(slide_layout)
            title = slide.shapes.title
            title.text = f"Lab Results - {lab_data.get('date', '')}"
            
            rows = len(lab_data['table']) + 1 #헤더를 포함한 행수
            cols = len(headers) #헤더의 수
            table = slide.shapes.add_table(rows, cols, Inches(0.5), Inches(1.5), Inches(5), Inches(3.0)).table
            #헤더 채우기
            for col_idx, header in enumerate(headers):
                cell = table.cell(0, col_idx)
                cell.text = header
                for paragraph in cell.text_frame.paragraphs:
                    for run in paragraph.runs:
                        run.font.size = Pt(5) 
            #데이터 채우기
            for row_idx, (_, data) in enumerate(lab_data['table'].iterrows()):
                for col_idx, header in enumerate(headers):
                    cell = table.cell(row_idx + 1, col_idx)
                    cell.text = str(data[header])
                    for paragraph in cell.text_frame.paragraphs:
                        for run in paragraph.runs:
                            run.font.size = Pt(5) 

def main_presentation():
    # PDF 파일 경로 설정
    pdf_path = "/Users/sinequanon/Documents/PerfectS/data/pdf_data/TEST_05.pdf"
    # 변수 들고오기
    soap_result, vital_check_data_list, lab_data_list, animal_data = main(pdf_path)
    print(soap_result, vital_check_data_list, lab_data_list, animal_data)

    
    pptx_file_path = '/Users/sinequanon/Documents/PerfectS/data/template.pptx'
    if not os.path.exists(pptx_file_path):
        print("PPTX file does not exist.")
        return

    prs = Presentation(pptx_file_path)

    # 이미지가 있는 경로
    image_directory = '/Users/sinequanon/Documents/PerfectS/data/images'
    images = [os.path.join(image_directory, f) for f in os.listdir(image_directory) if os.path.isfile(os.path.join(image_directory, f)) and f.endswith(('png', 'jpg', 'jpeg', 'gif', 'bmp'))]


    #특정 텍스트를 포함하는 슬라이드 찾기
    slide_layout = None
    slide_index = None
    for i, slide in enumerate(prs.slides):
        slide_title_shape = select_shape_by_text(slide, "Chart Image")
        if slide_title_shape:
            slide_layout = slide.slide_layout
            slide_index = i
            break

#이미지 추가
    if slide_layout:
        slide = prs.slides[slide_index]
        for i in range(0, len(images), 2):
            if i != 0:
                slide = copy_slide(prs, slide_index)
            slide.shapes.add_picture(images[i], Cm(2), Cm(5), width=Cm(10), height=Cm(7))
            if i + 1 < len(images):
                slide.shapes.add_picture(images[i + 1], Cm(14), Cm(5), width=Cm(10), height=Cm(7))
#테이블 추가하기
    vital_headers = ['날짜', '시간', 'BW(Kg)', 'BT(C)', 'BP(mmHg)', 'HR(/min)', 'Sign']
    lab_headers = ['검사명', '결과값', '단위', 'MIN', 'MAX', 'Description']

    add_vital_check_table_to_slide(prs, vital_headers, vital_check_data_list.to_dict('records'))
    add_lab_table_to_slide(prs, lab_headers, lab_data_list)

#signalment_data가 포함되어 있으면 animal 데이터를 넣는다
    for slide in prs.slides:
        animal_slide = select_shape_by_text(slide, "Signalment_data")
        if animal_slide:
            insert_data(slide, animal_data)


    #해당 텍스트가 포함되어 있으면 soap_data를 넣는다         
    texts_to_find = ["Subjective text", "Objective text", "Assessment text", "Plan text"]
    for slide in prs.slides:
        soap_shapes = select_shapes_by_texts(slide, texts_to_find)
        for soap_shap in soap_shapes:
            for soap_data in soap_result:
                insert_data(slide, soap_data)
                
    output_path = '/Users/sinequanon/Documents/PerfectS/data/pptx/test05.pptx'
    prs.save(output_path)

if __name__ == "__main__":
    main_presentation()



[{'date': '2023-03-13 15:02', 'subjective': ['Subjective', 'CC: 좌측 후지파행 Rad(E)', '[초진] 좌측 후지파행', 'S) 주호소)', '어렸을때 좌측 후지 수술했는데 어떤수술인지는 모르심', '> 우측 fhno 한 것으로 확인', '수술후에는 잘 걸었음', '1~2개월전부터 좌측 후지 힘을 덜주는것 같음', '2주전 목욕 맡기고나서 조금 심해짐', '히스토리)', '1. 복용중인 약:- 2. 접종이력: 3. 수술(마취 이력): 8개월쯤 우측 fhno 4. 사고/중독이력: 5. 실내/실외: 6. 사료:', '금식:', '전신상태)', '1. 식이, 음수, 배변, 배뇨: 2. 피부: 3. 호흡: 4. 근육/관절: 슬개골 탈구 좌측 3기, 우측 2기 5. 심장: 6. 방광: 7. 기타:', '검사계획)', '신체검사, 신경계평가 , CBC, 혈청, 가스분석, 복부초음파 , 심초음파, 뇨검사, CT, MRI, 세침흡인검사 , 생검', '검사전 처치)', '수액, 산소, 진통, 이뇨제, CPR', 'O)', '신체검사)', '1. BCS: 3/9 2. 심박 : 호흡수 : 3. 수화상태 : 4. 체표림프절 : 5. 청진:nrf 6. 혈압 : 7. 슬개골탈구 : 8. 방광촉진: 9. ETC.:', '혈액검사(주요소견만 )', '1. CBC 2. Biochemistry 1. 3. Blood gas 1.', '영상검사)', '방사선 : bilateral MPL, 양쪽 관절 부종 소견 명확하지않음', 'Rad)Hindlimb#름 - 좌측 고관절 subluxation, 경미한 DJD - 양측 MPL - 양측 근육 위축', '심전도)', '1.', '신체검사)', '보행시 경미한 좌측 후지 파행', '서있을때 힘을 덜주는 양상', 'back pain-', '후지신경반응 정상', '장요근 통증-', '슬개골 내측 탈구 Lt 3기 Rt 2기(왼쪽은 탈구시 체중지지 잘 못함)', 'drawer Lt+, Rt-', '고관절 rom